# Kenya Gazette PDF to structured JSON (Docling)

This notebook runs **Docling** on one or more Gazette PDFs and builds a **single JSON-friendly record per file** with:

- PDF metadata: inferred title, file name, size in bytes, page count
- Full **Docling** exports: plain text, Markdown, and a compact document summary (full `export_to_dict` is optional — it can be large)
- **Gazette notices** split by the phrase `GAZETTE NOTICE NO.` (Kenya-style), with notice number, header line, and full notice text

**Input:** drop PDFs into the `pdfs/` folder next to this notebook (or change `PDF_DIR` below). Use **Choose which PDFs** to run on every file or only a chosen subset.

## Environment

Use the project virtual environment (already created in this folder as `.venv`):

- **Windows (PowerShell):** `.\.venv\Scripts\Activate.ps1`
- **Then:** `python -m pip install -r requirements.txt` if needed
- **Jupyter kernel:** register with `python -m ipykernel install --user --name=gazette-docling --display-name="Python (gazette-docling)"` using the venv’s `python`.

Select that kernel in this notebook before running cells.

In [1]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from docling.document_converter import DocumentConverter
from docling_core.types.doc.labels import DocItemLabel

# Project root = folder containing this notebook
PROJECT_ROOT = Path.cwd().resolve()
PDF_DIR = PROJECT_ROOT / "pdfs"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF_DIR:", PDF_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PDF_DIR: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\pdfs
OUTPUT_DIR: C:\Users\Ronald Wahome\Documents\gazette_test_inputs\output


## Gazette notice splitting

Notices are detected by a case-insensitive match on **`GAZETTE NOTICE NO.`** followed by an optional notice identifier (digits, letters, slashes, hyphens). Each segment runs until the next such marker or end of document.

You can tune `NOTICE_PATTERN` if your issues use a different wording.

In [2]:
# Match the start of each notice; group 1 = notice number / reference (best effort)
NOTICE_PATTERN = re.compile(
    r"(?is)\bGAZETTE\s+NOTICE\s+NO\.?\s*[:\-]?\s*([0-9A-Za-z\/\-]*)"
)


def split_gazette_notices(full_text: str) -> list[dict[str, Any]]:
    """Split flat text into notice blocks after 'GAZETTE NOTICE NO.'."""
    if not full_text or not full_text.strip():
        return []

    matches = list(NOTICE_PATTERN.finditer(full_text))
    if not matches:
        return [
            {
                "gazette_notice_no": None,
                "gazette_notice_header": None,
                "gazette_notice_full_text": full_text.strip(),
                "other_attributes": {
                    "reason": "no GAZETTE NOTICE NO. markers found; entire text returned as one block",
                },
            }
        ]

    notices: list[dict[str, Any]] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        block = full_text[start:end].strip()
        raw_no = (m.group(1) or "").strip()
        first_line, _, rest = block.partition("\n")
        header_candidate = first_line.strip()
        if len(header_candidate) > 500:
            header_candidate = header_candidate[:500] + "…"

        notices.append(
            {
                "gazette_notice_no": raw_no or None,
                "gazette_notice_header": header_candidate,
                "gazette_notice_full_text": block,
                "other_attributes": {
                    "char_span_start": start,
                    "char_span_end": end,
                    "lines_after_header": rest.count("\n") + (1 if rest else 0),
                },
            }
        )
    return notices


def extract_title_from_docling(doc) -> str:
    for item in getattr(doc, "texts", []) or []:
        if getattr(item, "label", None) == DocItemLabel.TITLE and getattr(item, "text", None):
            return str(item.text).strip()
    return ""


def docling_export_summary(doc_dict: dict[str, Any]) -> dict[str, Any]:
    """Small fingerprint of the Docling JSON without dumping huge trees twice."""
    texts = doc_dict.get("texts") or []
    return {
        "schema_name": doc_dict.get("schema_name"),
        "version": doc_dict.get("version"),
        "name": doc_dict.get("name"),
        "texts_count": len(texts) if isinstance(texts, list) else None,
        "tables_count": len(doc_dict.get("tables") or []) if isinstance(doc_dict.get("tables"), list) else None,
        "pictures_count": len(doc_dict.get("pictures") or []) if isinstance(doc_dict.get("pictures"), list) else None,
        "pages_count": len(doc_dict.get("pages") or []) if isinstance(doc_dict.get("pages"), list) else None,
    }

## Convert pipeline

`DocumentConverter` is created once and reused. For each PDF we populate the output record and optionally write the full Docling JSON to `output/<stem>_docling.json`.

In [3]:
@dataclass
class GazettePipeline:
    converter: DocumentConverter = field(default_factory=DocumentConverter)
    include_full_docling_dict: bool = False

    def process_pdf(self, pdf_path: Path) -> dict[str, Any]:
        pdf_path = Path(pdf_path).resolve()
        stat = pdf_path.stat()

        result = self.converter.convert(str(pdf_path))
        doc = result.document

        title_guess = extract_title_from_docling(doc) or pdf_path.stem.replace("_", " ")
        page_count = len(doc.pages) if getattr(doc, "pages", None) else None

        plain = doc.export_to_text()
        md = doc.export_to_markdown()

        doc_dict = doc.export_to_dict()

        record: dict[str, Any] = {
            "pdf_title": title_guess,
            "pdf_file_name": pdf_path.name,
            "pdf_path": str(pdf_path),
            "pdf_size_bytes": stat.st_size,
            "pages": page_count,
            "docling": {
                "export_summary": docling_export_summary(doc_dict),
                "full_markdown": md,
                "full_plain_text": plain,
            },
            "gazette_notices": split_gazette_notices(plain),
        }

        if self.include_full_docling_dict:
            record["docling"]["full_docling_document_dict"] = doc_dict

        out_json = OUTPUT_DIR / f"{pdf_path.stem}_gazette.json"
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        docling_only = OUTPUT_DIR / f"{pdf_path.stem}_docling.json"
        with open(docling_only, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

        print(f"Wrote: {out_json}")
        print(f"Wrote: {docling_only}")
        return record


def run_pdfs(pipeline: GazettePipeline, pdf_paths: list[Path]) -> list[dict[str, Any]]:
    """Run the pipeline on an explicit ordered list of PDF paths."""
    if not pdf_paths:
        print("No PDF paths to process.")
        return []
    results: list[dict[str, Any]] = []
    for p in pdf_paths:
        print("\n--- Processing:", p.name, "---")
        results.append(pipeline.process_pdf(p))
    return results


def run_folder(pipeline: GazettePipeline, folder: Path | None = None) -> list[dict[str, Any]]:
    """Process every *.pdf in folder (same as selection mode \"all\")."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    return run_pdfs(pipeline, pdfs)


def resolve_pdf_selection(
    mode: str,
    selected_names: list[str],
    folder: Path | None = None,
) -> list[Path]:
    """Resolve \"all\" or \"selected\" to a list of existing PDF paths under folder."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    m = (mode or "all").strip().lower()
    if m == "all":
        return pdfs
    if m == "selected":
        if not selected_names:
            print("PDF_SELECTION_MODE is 'selected' but SELECTED_PDF_NAMES is empty. Nothing to do.")
            return []
        by_name = {p.name: p for p in pdfs}
        out: list[Path] = []
        missing: list[str] = []
        for raw in selected_names:
            name = raw.strip()
            if not name:
                continue
            p = by_name.get(name)
            if p is None:
                p = by_name.get(Path(name).name)
            if p is None:
                missing.append(name)
            elif p not in out:
                out.append(p)
        if missing:
            print("Warning: not found in", folder, ":", missing)
            print("Available PDFs:", [p.name for p in pdfs])
        return out
    raise ValueError('PDF_SELECTION_MODE must be "all" or "selected".')

## Choose which PDFs to process

Set **`PDF_SELECTION_MODE`** in the next cell:

- **`"all"`** — every `*.pdf` under `PDF_DIR`
- **`"selected"`** — only files listed in **`SELECTED_PDF_NAMES`** (exact file names as they appear in `pdfs/`)

Then run the pipeline cell.

## Run

First conversion may **download models** and take longer. Subsequent runs are faster.

In [ ]:
# --- Configure which PDFs to process ---
# When mode is "selected", list exact file names as they appear in PDF_DIR (order is preserved).
# When mode is "all", SELECTED_PDF_NAMES is ignored.
PDF_SELECTION_MODE = "selected"  # "all" | "selected"
SELECTED_PDF_NAMES: list[str] = [
    "Kenya Gazette Vol CXINo 110.pdf",
]

pipeline = GazettePipeline(include_full_docling_dict=False)
pdf_paths = resolve_pdf_selection(PDF_SELECTION_MODE, SELECTED_PDF_NAMES)
print(
    "Mode:",
    PDF_SELECTION_MODE,
    "|",
    len(pdf_paths),
    "file(s):",
    [p.name for p in pdf_paths],
)
results = run_pdfs(pipeline, pdf_paths)

if results:
    sample = results[0]
    print("Keys:", list(sample.keys()))
    print("Pages:", sample.get("pages"))
    print("Notice count:", len(sample.get("gazette_notices") or []))
    display_snippet = json.dumps(sample, ensure_ascii=False, indent=2)
    if len(display_snippet) > 8000:
        print(display_snippet[:8000] + "\n... [truncated for display; see output JSON files] ...")
    else:
        print(display_snippet)

## Optional: embed full Docling tree in the main record

Set `include_full_docling_dict=True` if you need the complete `DoclingDocument` as JSON inside `gazette.json` (large).

In [ ]:
# Example: toggle full dict for a single file
# p = PDF_DIR / "your_issue.pdf"
# if p.exists():
#     full_pipeline = GazettePipeline(include_full_docling_dict=True)
#     full_pipeline.process_pdf(p)